# 17. Mixed-data pipelines
Real tables often mix numbers and categories. ColumnTransformer sends selected columns through different preprocessing branches, then joins their outputs. Pipeline connects that prepared table to a predictor. The resulting object accepts raw records with the same input contract.

Use a numeric branch for imputation and scaling and a categorical branch for imputation and encoding. Inspect generated feature names. set_output(transform="pandas") can help retain names, but pandas output is incompatible with sparse output from OneHotEncoder unless a compatible dense configuration is selected. Use nested names such as model__C for tuning a named pipeline step.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
X = pd.DataFrame({"size": [1, 2, 3, 4], "city": ["A", "B", "A", "B"]}) # Raw mixed inputs.
y = [10, 22, 30, 42] # Numerical targets.
prep = ColumnTransformer([("num", SimpleImputer(), ["size"]), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["city"])]) # Define both branches.
prep.set_output(transform="pandas") # Preserve generated names in a table.
model = make_pipeline(prep, LinearRegression()).fit(X, y) # Fit all steps together.
print("Features:", prep.get_feature_names_out().tolist()) # See the joined representation.
print("New prediction:", model.predict(pd.DataFrame({"size": [5], "city": ["C"]})).round(2)) # Accept an unknown category.
assert len(model.predict(X)) == 4 # Predict once per raw input row.

## Inspect named features without changing global state
Named outputs make pipeline debugging concrete: inspect which columns each branch produced and whether a target or identifier slipped in. set_config(transform_output="pandas") changes the global default. config_context applies the same option only within a block and restores the previous setting. Sparse one-hot output is incompatible with a pandas output request in this configuration; explicitly choose dense output only when its memory cost is acceptable. Keep transformations inside the evaluation boundary even while inspecting names.

In [ ]:
import pandas as pd
from sklearn import config_context
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
X = pd.DataFrame({"hours": [1., 2., 3.], "city": ["A", "B", "A"]}) # Labeled raw inputs.
prep = ColumnTransformer([("num", StandardScaler(), ["hours"]), ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["city"])]) # Two named branches.
with config_context(transform_output="pandas"): # Scope the output setting instead of changing the whole session.
    Z = prep.fit_transform(X) # Inspect training output in a named table.
    print("Columns:", Z.columns.tolist()) # Verify column names and order.
    print("Names agree:", Z.columns.tolist() == prep.get_feature_names_out().tolist()) # Compare both inspection APIs.
assert len(prep.get_feature_names_out()) == 3 # One scaled feature and two category indicators.

### Reading the result
Prefixes identify the preprocessing branch. DataFrame outputs assist inspection, but input dtypes and category meanings still need checking. Returning names is not proof the model’s inputs were available at prediction time.

### Transfer exercise
Inspect a fitted mixed-data pipeline and list the raw input columns, transformed feature names, and final estimator width.

## Your turn
Build separate named steps prep and model with Pipeline. Write the parameter name for the classifier’s regularization setting C.

Try the task before opening the solution on the course website.